In [3]:
!pip install ipyplot
!pip install torchvision
!pip install diffusers
import os 
import torchvision
import torchvision.transforms as transforms
from tqdm import tqdm
import torch
from transformers import CLIPModel, CLIPTextModel, CLIPTokenizer
from diffusers import AutoencoderKL, UNet2DConditionModel
import numpy as np

In [4]:
vae = AutoencoderKL.from_pretrained("stabilityai/sd-vae-ft-ema").to("cuda")
print("done")

AssertionError: Torch not compiled with CUDA enabled

In [ ]:
import os
from torchvision.datasets import ImageFolder
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

In [ ]:
from PIL import Image
import urllib
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((128, 128)),    
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])
print("done")

In [ ]:
from transformers import CLIPTextModel, CLIPTokenizer
model = CLIPTextModel.from_pretrained("openai/clip-vit-base-patch32")
tokenizer = CLIPTokenizer.from_pretrained("openai/clip-vit-base-patch32")

In [ ]:
import json
from PIL import Image
import os
import numpy as np
from tqdm import tqdm

In [ ]:
categories_path = '/kaggle/input/coco-2017-dataset/coco2017/annotations/captions_train2017.json'  # Adjust the path
with open(categories_path, 'r') as f:
    categories_data = json.load(f)

In [ ]:
categories_data.keys()

In [ ]:
categories_data['annotations'][0]

In [ ]:
len(categories_data['annotations'])

In [ ]:
data_dict = {1 : []}

In [ ]:
Image.open("/kaggle/input/coco-2017-dataset/coco2017/train2017/000000501175.jpg")

In [ ]:
with torch.no_grad():
    for s in tqdm(os.listdir('/kaggle/input/coco-2017-dataset/coco2017/train2017')):
        try:
            image = np.array(Image.open("/kaggle/input/coco-2017-dataset/coco2017/train2017/" + s))
            image = transform(image).to("cuda")
            image = image.unsqueeze(0)
            image = vae.encode(image.float()).latent_dist.sample().detach().cpu().permute(0, 2, 3, 1).numpy()
            index = int(s[0:12])
            data_dict[index] = []
            data_dict[index].append(image)
        except:
#             print("error")
            continue

In [ ]:
with torch.no_grad():
    for s in tqdm(os.listdir('/kaggle/input/coco-2017-dataset/coco2017/test2017')):
        try:
            image = np.array(Image.open("/kaggle/input/coco-2017-dataset/coco2017/test2017/" + s))
            image = transform(image).to("cuda")
            image = image.unsqueeze(0)
            image = vae.encode(image.float()).latent_dist.sample().detach().cpu().permute(0, 2, 3, 1).numpy()
            index = int(s[0:12])
            data_dict[index] = []
            data_dict[index].append(image)
        except:
#             print("error")
            continue

In [ ]:
type(categories_data['annotations'])

In [ ]:
with torch.no_grad():
    for text in tqdm(categories_data['annotations']):
        index = text['image_id']
        text = text['caption']
        if index in data_dict:
            try:
                inputs = tokenizer(text, padding=True, truncation=True, return_tensors="pt")
                text_embeddings = model(**inputs).pooler_output
                data_dict[index].append(text_embeddings)
                
            except:
                data_dict.pop(index)

In [ ]:
data_dict.pop(1)

In [ ]:
data_dict[501175][1]

In [ ]:
image_latents= []
caption_latents = []
e = 0
for key in data_dict.keys():
    try:

        image_latents.append(data_dict[key][0])
        caption_latents.append(data_dict[key][1])
    except:
        e+= 1
        image_latents.pop()
#         print(key)

In [ ]:
e

In [ ]:
len(data_dict.keys())

In [ ]:
len(image_latents)

In [ ]:
image_latents = np.vstack(image_latents)
caption_latents = np.vstack(caption_latents)

In [ ]:
image_latents.shape

In [ ]:
caption_latents.shape

In [ ]:
np.save("/kaggle/working/image_latents.npy", image_latents)
np.save("/kaggle/working/text_embeddings.npy", caption_latents)